# Language Processing Laboratory — Session 04
## Persian Stopwords, Stemming, Lemmatization, and Dictionary-Based Spelling Correction

**Duration:** 120 minutes  
**Level:** Undergraduate  
**Prerequisite:** Sessions 01–03, basic Python, regular expressions, and word tokenization  
**Required libraries:** Python standard library; JupyterLab is used to run this notebook.

This starter lab applies language-processing concepts to Persian examples. It distinguishes task-aware stopword filtering, rule-based stemming, lexicon-based lemmatization, and spelling correction using Levenshtein edit distance. The provided corpus and lexicon are small teaching resources, not general-purpose Persian language resources.

## Lab workflow and assessment

| Time | Activity |
|---:|---|
| 0–10 min | Environment, normalization, and miniature lexicon |
| 10–30 min | Stopwords: purpose, risks, implementation, checks |
| 30–48 min | Rule-based stemming and error analysis |
| 48–65 min | Lemmatization and comparison with stemming |
| 65–100 min | Levenshtein distance and ranked spelling correction |
| 100–115 min | Evaluate examples; inspect false corrections and edge cases |
| 115–120 min | Exit ticket and submission checklist |

The corpus and dictionary are intentionally small and mostly synthetic. Passing the public checks demonstrates that the implementation follows this lab's specification; it does not establish general performance on Persian text.

---
# 0. Environment, normalization, and the mini-lexicon

Run this notebook from `sessions/04_morphology_correction/` so the relative paths under `data/` resolve. The normalization helper is supplied: it NFC-normalizes text and maps common Arabic forms of yeh/kaf to Persian forms. It deliberately **preserves ZWNJ (`U+200C`)**. This is a narrow classroom policy, not a claim that every Persian text should be normalized identically.

In [ ]:
import re
import sys
import unicodedata
from collections import Counter
from pathlib import Path

print('Python:', sys.version.split()[0])
print('Working directory:', Path.cwd())
DATA_DIR = Path('data')
STOPWORDS_PATH = DATA_DIR / 'stopwords_fa.txt'
LEXICON_PATH = DATA_DIR / 'mini_lexicon_fa.tsv'
assert STOPWORDS_PATH.is_file(), f'Missing {STOPWORDS_PATH}; open the notebook from its session folder.'
assert LEXICON_PATH.is_file(), f'Missing {LEXICON_PATH}; keep the data folder beside this notebook.'


def normalize_persian(text):
    """Apply the lab's conservative Unicode/Persian character policy; preserve ZWNJ."""
    text = unicodedata.normalize('NFC', str(text))
    return text.replace('ي', 'ی').replace('ى', 'ی').replace('ك', 'ک')


PERSIAN_LETTERS = r'\u0621-\u063A\u0641-\u064A\u067E\u0686\u0698\u06A9\u06AF\u06CC'
WORD_PATTERN = re.compile(
    rf'[{PERSIAN_LETTERS}]+(?:\u200c[{PERSIAN_LETTERS}]+)*|[A-Za-z]+|[0-9۰-۹٠-٩]+'
)


def tokenize_words(text):
    """Extract lexical tokens for this lab; punctuation is excluded, internal ZWNJ is kept."""
    return WORD_PATTERN.findall(normalize_persian(text))


print('Normalization example:', normalize_persian('يک كتاب می‌روم'))
print('Tokenization example:', tokenize_words('این کتاب‌ها در دانشگاه است.'))

In [ ]:
# TODO: enter your own metadata before submission.
student_id = ''
student_name = ''
university = ''
print('Student ID:', student_id)
print('Name:', student_name)
print('University:', university)

In [ ]:
# Load the supplied, task-specific stopword list and the tiny frequency/lemma lexicon.
STOPWORDS = {
    normalize_persian(line.strip())
    for line in STOPWORDS_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip() and not line.lstrip().startswith('#')
}

LEXICON = {}
WORD_FREQUENCIES = {}
for line_number, line in enumerate(LEXICON_PATH.read_text(encoding='utf-8').splitlines(), start=1):
    if line_number == 1:  # Header: word<TAB>frequency<TAB>lemma
        continue
    if not line.strip():
        continue
    word, frequency, lemma = line.split('\t')
    word = normalize_persian(word)
    lemma = normalize_persian(lemma)
    LEXICON[word] = lemma
    WORD_FREQUENCIES[word] = int(frequency)

print(f'{len(STOPWORDS)} stopword examples loaded.')
print(f'{len(WORD_FREQUENCIES)} lexicon entries loaded.')
print('Negation terms intentionally not listed as stopwords:', {'نه', 'نیست', 'نمی', 'بدون'} - STOPWORDS)

---
# 1. Persian stopwords: what they are and why we might remove them

**Stopwords** are tokens a particular pipeline chooses to ignore because they may be frequent and relatively uninformative for that task. Common Persian examples can include `و`، `از`، `به`، `در`، `را`، `که` and `این`.

### Why remove stopwords?

- In some **bag-of-words search or document-classification baselines**, very frequent function words can occupy feature space while contributing little to the particular decision.
- Removing them can reduce vocabulary size and make term-frequency comparisons easier to inspect.
- A stopword list can also be used as a transparent, reproducible preprocessing policy.

### Why not remove them automatically?

- They carry grammar and relations between words; removal can harm parsing, translation, question answering and other tasks.
- Small words may be semantically crucial. Removing `نه` or `نیست`, or the negating prefix in `نمی‌روم`, can reverse the meaning of a sentence.
- A word that is unhelpful for one task may be useful for another. There is no universally correct Persian stopword list.
- Stopword removal is generally not a default step for contextual models such as BERT; evaluate it rather than assuming it helps.

The supplied `data/stopwords_fa.txt` is a small teaching list, **not an authoritative or exhaustive Persian stopword resource**. It intentionally excludes negation terms. The supplied tokenizer keeps `می‌روم` as one token so we do not accidentally strip the internal `می` substring.

In [ ]:
DEMO_TEXTS = [
    'این فیلم خوب است',
    'این فیلم بد نیست',
    'من به دانشگاه می‌روم',
]

for text in DEMO_TEXTS:
    print('Text:   ', text)
    print('Tokens: ', tokenize_words(text))
    print()


### ✍️ Exercise 1 — Implement `remove_stopwords`

Implement `remove_stopwords(tokens, stopwords=STOPWORDS)`.

Requirements:

1. Normalize each token with `normalize_persian` before comparing it with the stopword set.
2. Remove a token only when the **entire token** is in the stopword set; do not remove matching substrings.
3. Preserve token order and all non-stopword tokens, including `نه`, `نیست`, `نمی‌روم` and `بدون`.
4. Return a new list; do not mutate the caller's list.
5. Do not change `STOPWORDS` to make tests pass. If you propose changing the list, first explain the task and evidence that justify that change.

In [ ]:
def remove_stopwords(tokens, stopwords=STOPWORDS):
    """Return tokens except exact normalized stopword matches. TODO: Exercise 1."""
    # TODO: implement exact-token filtering; preserve order and do not mutate tokens.
    return list(tokens)


def check_stopword_removal(remove_fn, *, verbose=True):
    cases = [
        (['این', 'کتاب', 'در', 'دانشگاه', 'است'], ['کتاب', 'دانشگاه']),
        (['نه', 'خوب', 'نیست', 'این'], ['نه', 'خوب', 'نیست']),
        (['نمی‌روم', 'بدون', 'کتاب', 'و', 'دفتر'], ['نمی‌روم', 'بدون', 'کتاب', 'دفتر']),
        (['يك', 'كتاب', 'و'], ['كتاب']),  # normalize for comparison; preserve surviving surface spelling
    ]
    passed = 0
    for index, (tokens, expected) in enumerate(cases, 1):
        try:
            got = remove_fn(tokens)
            ok = got == expected
        except Exception as exc:
            got, ok = repr(exc), False
        passed += int(ok)
        if verbose:
            print(f'{"PASS" if ok else "FAIL"} stopword case {index}: expected={expected!r}; got={got!r}')
    if verbose:
        print(f'{passed}/{len(cases)} stopword cases passed')
    return passed, len(cases)

check_stopword_removal(remove_stopwords)

In [ ]:
# Compare the result after completing Exercise 1.
for text in DEMO_TEXTS:
    tokens = tokenize_words(text)
    print('Text:      ', text)
    print('Tokens:    ', tokens)
    print('Filtered:  ', remove_stopwords(tokens))
    print()


### ✍️ Exercise 1b — Evidence and task choice

1. Count tokens before and after filtering for each demonstration sentence.
2. Which sentence would be dangerous to process with a stopword list that removes negation?
3. Name one task where stopword removal is likely to be inappropriate and one task where it may help.
4. In 3–5 sentences, explain why you would keep the stopword list versioned with an NLP experiment.

---
# 2. Stemming: a deliberately simple suffix baseline

A **stemmer** applies surface-form rules to produce a shortened stem; that output need not be a valid dictionary word or the true lemma. For this activity we use a small, conservative suffix list and a minimum remaining-root length. It is a teaching baseline, not a complete Persian morphological analyzer.

Examples of suffixes that may be stripped in this limited exercise include plural `ها` / `‌ها`, possessive/plural forms such as `‌های`, comparative `تر`, superlative `ترین`, and a limited human-plural ending such as `یان`. These rules can over-stem or under-stem: a suffix-like sequence may be part of a lexical root, and Persian morphology is more complex than a few string endings.

In [ ]:
STEM_SUFFIXES = sorted(
    ['‌هایشان', 'هایشان', '‌هایمان', 'هایمان', '‌های', 'های', '‌ها', 'ها', 'ترین', 'تر', 'یان'],
    key=len,
    reverse=True,
)
MIN_STEM_LENGTH = 3
print('Longest-first suffix order:', STEM_SUFFIXES)

### ✍️ Exercise 2 — Implement `simple_persian_stem`

Write `simple_persian_stem(word)` using `STEM_SUFFIXES` and `MIN_STEM_LENGTH`.

- Normalize the input using `normalize_persian`.
- Check suffixes longest first so `ترین` is considered before `تر`.
- Remove at most **one** matching suffix per call in this lab.
- Strip it only if the remaining stem has at least `MIN_STEM_LENGTH` characters.
- If no suffix qualifies, return the normalized word unchanged.

This is a rule-based demonstration. Do not describe its output as a gold-standard linguistic stem.

In [ ]:
def simple_persian_stem(word):
    """Apply one longest matching suffix rule. TODO: Exercise 2."""
    word = normalize_persian(word)
    # TODO: loop over STEM_SUFFIXES; enforce MIN_STEM_LENGTH; return a stem or word.
    return word


def check_stemming(stem_fn, *, verbose=True):
    cases = [
        ('کتاب‌ها', 'کتاب'),
        ('کتاب‌های', 'کتاب'),
        ('کتابها', 'کتاب'),
        ('دانشجویان', 'دانشجو'),
        ('زیباترین', 'زیبا'),
        ('ایران', 'ایران'),
        ('درخت', 'درخت'),
    ]
    passed = 0
    for index, (word, expected) in enumerate(cases, 1):
        try:
            got = stem_fn(word)
            ok = got == expected
        except Exception as exc:
            got, ok = repr(exc), False
        passed += int(ok)
        if verbose:
            print(f'{"PASS" if ok else "FAIL"} stem case {index}: {word!r} -> expected {expected!r}, got {got!r}')
    if verbose:
        print(f'{passed}/{len(cases)} stemming cases passed')
    return passed, len(cases)

check_stemming(simple_persian_stem)

### ✍️ Exercise 2b — Inspect stemming errors

Test the stemmer on at least five additional Persian words, including one proper noun and one word whose final letters resemble a suffix but are part of the root. Record:

- the original word and predicted stem;
- whether the output remains a valid word;
- one example of over-stemming or under-stemming;
- one rule you would change, with a warning about possible side effects.

---
# 3. Lemmatization: map a surface form to a lexical lemma

**Lemmatization** aims to map inflected forms to a lexically meaningful base form, usually with language-specific morphology and lexical information. Unlike a suffix-only stemmer, a lemmatizer may map several surface forms to one lemma even when a simple suffix rule cannot do so. For example, this tiny classroom lexicon maps `رفتم` and `می‌روم` to `رفتن`.

The notebook's `mini_lexicon_fa.tsv` is intentionally small, so the implementation below is **dictionary-based**. If a word is not in the lexicon, the fallback is its normalized surface form. A production lemmatizer needs broad lexical coverage and may also use part-of-speech or morphological context. Lemmas can lose grammatical information, so keep the original surface text if that information matters.

### ✍️ Exercise 3 — Implement `lemmatize_word`

Implement `lemmatize_word(word, lexicon=LEXICON)`:

1. normalize the input;
2. return the dictionary lemma if the normalized surface form exists in `lexicon`;
3. otherwise return the normalized surface form unchanged.

Do not apply the stemmer as a fallback: this is a dictionary lookup exercise, and a failed lookup must remain visible rather than silently inventing a lemma.

In [ ]:
def lemmatize_word(word, lexicon=LEXICON):
    """Return a dictionary lemma or the normalized surface form. TODO: Exercise 3."""
    # TODO: normalize, perform one exact lookup, and preserve unknown words.
    return normalize_persian(word)


def check_lemmatization(lemma_fn, *, verbose=True):
    cases = [
        ('کتاب‌ها', 'کتاب'),
        ('دانشجویان', 'دانشجو'),
        ('می‌رود', 'رفتن'),
        ('رفتم', 'رفتن'),
        ('زیباترین', 'زیبا'),
        ('واژه‌ناشناخته', 'واژه‌ناشناخته'),
        ('يك', 'یک'),  # normalizes even when there is no lemma entry
    ]
    passed = 0
    for index, (word, expected) in enumerate(cases, 1):
        try:
            got = lemma_fn(word)
            ok = got == expected
        except Exception as exc:
            got, ok = repr(exc), False
        passed += int(ok)
        if verbose:
            print(f'{"PASS" if ok else "FAIL"} lemma case {index}: {word!r} -> expected {expected!r}, got {got!r}')
    if verbose:
        print(f'{passed}/{len(cases)} lemmatization cases passed')
    return passed, len(cases)

check_lemmatization(lemmatize_word)

In [ ]:
# Compare stemming with dictionary lemmatization after completing Exercise 3.
comparison_words = ['کتاب‌ها', 'زیباترین', 'می‌رود', 'رفتم', 'ایران']
for word in comparison_words:
    print(f'{word} | stem={simple_persian_stem(word)} | lemma={lemmatize_word(word)}')


### Optional extension — compare with Hazm

[Hazm](https://github.com/roshan-research/hazm) is a Persian NLP toolkit with stemming and lemmatization support. It is **optional**; the core lab requires only the Python standard library. If Hazm is installed in your environment, run the guarded cell below and compare its behavior with the small hand-written baseline. Do not expect the teaching lexicon and Hazm to give identical outputs for every form.

In [ ]:
try:
    from hazm import Stemmer, Lemmatizer
except ImportError:
    print('Hazm is not installed; skipping optional comparison. Core exercises do not require it.')
else:
    hazm_stemmer = Stemmer()
    hazm_lemmatizer = Lemmatizer()
    for word in ['کتاب‌ها', 'دانشجویان', 'می‌رود', 'رفتم', 'زیباترین']:
        print(word, '| Hazm stem:', hazm_stemmer.stem(word), '| Hazm lemma:', hazm_lemmatizer.lemmatize(word))

---
# 4. Levenshtein edit distance

**Levenshtein distance** is the minimum number of single-character insertions, deletions, or substitutions needed to transform one string into another. In this lab, compare normalized Unicode character sequences (Python code points), not UTF-8 bytes. A transposition of adjacent characters is normally two edits in standard Levenshtein distance.

For strings `a` and `b`, dynamic programming uses the recurrence:

- `D[i, 0] = i` and `D[0, j] = j`;
- `D[i, j]` is the minimum of deletion `D[i-1, j] + 1`, insertion `D[i, j-1] + 1`, and substitution `D[i-1, j-1] + cost`;
- `cost = 0` when the current characters match, otherwise `1`.

The final value `D[len(a), len(b)]` is the distance. Normalization policy matters: if Arabic and Persian character variants are normalized to the same form, they should not count as spelling errors for this exercise.

### ✍️ Exercise 4 — Implement `levenshtein_distance`

Implement the dynamic-programming algorithm. Use a one-row optimization if you can explain it; a full matrix is also acceptable. Requirements:

- normalize both inputs first;
- return `0` for equal strings;
- support empty strings;
- count insertions, deletions, and substitutions, but not adjacent transpositions as one operation;
- do not compare UTF-8 byte sequences.

In [ ]:
def levenshtein_distance(left, right):
    """Return character-level Levenshtein distance. TODO: Exercise 4."""
    left = normalize_persian(left)
    right = normalize_persian(right)
    # TODO: implement dynamic programming using the recurrence above.
    return max(len(left), len(right))


def check_edit_distance(distance_fn, *, verbose=True):
    cases = [
        ('کتاب', 'کتاب', 0),
        ('کتاب', 'کباب', 1),
        ('دانشجوو', 'دانشجو', 1),
        ('داده', 'داد', 1),
        ('', 'فارسی', 5),
        ('يك', 'یک', 0),
        ('کتاب', 'کتبا', 2),  # swapping adjacent characters takes two standard edits
    ]
    passed = 0
    for index, (left, right, expected) in enumerate(cases, 1):
        try:
            got = distance_fn(left, right)
            ok = got == expected
        except Exception as exc:
            got, ok = repr(exc), False
        passed += int(ok)
        if verbose:
            print(f'{"PASS" if ok else "FAIL"} edit-distance case {index}: d({left!r}, {right!r}) expected {expected}, got {got!r}')
    if verbose:
        print(f'{passed}/{len(cases)} edit-distance cases passed')
    return passed, len(cases)

check_edit_distance(levenshtein_distance)

---
# 5. Dictionary-based Persian spelling correction

Spelling correction usually needs more than distance alone. This exercise uses the small lexicon in `data/mini_lexicon_fa.tsv` and ranks candidates by:

1. **smaller edit distance first**;
2. when distances tie, **higher corpus frequency first**;
3. lexical order as a deterministic final tie-break.

This is a simple baseline—not a production spell checker. It has no sentence context, keyboard model, morphology-aware candidate generation, or calibrated confidence. It can change a valid but rare name into a common word. The safe default is to suggest candidates for review rather than silently rewrite a document. The lexicon is deliberately tiny and hand-curated; absence from it is not evidence that a word is misspelled, and the frequency values are illustrative rather than corpus estimates.

In [ ]:
def suggest_corrections(word, word_frequencies=WORD_FREQUENCIES, max_distance=2, limit=5):
    """Rank non-exact candidates by (distance asc, frequency desc, spelling asc)."""
    normalized = normalize_persian(word)
    if normalized in word_frequencies:
        return []
    ranked = []
    for candidate, frequency in word_frequencies.items():
        distance = levenshtein_distance(normalized, candidate)
        if 0 < distance <= max_distance:
            ranked.append((candidate, distance, frequency))
    ranked.sort(key=lambda item: (item[1], -item[2], item[0]))
    return ranked[:limit]


for typo in ['دانشجوو', 'کتبا', 'ك', 'واژه‌ناشناخته']:
    print(f'{typo!r} -> {suggest_corrections(typo)}')

### ✍️ Exercise 5 — Implement `correct_word`

Implement `correct_word(word, word_frequencies=WORD_FREQUENCIES, max_distance=2)` using `suggest_corrections`.

- If the normalized word is already in the lexicon, return that normalized known form unchanged.
- Otherwise, return the highest-ranked suggestion if one exists within the maximum distance.
- If no suggestion qualifies, return the original input unchanged (do not invent a correction).
- Do not mutate the frequency dictionary.

In a real user-facing system, return suggestions plus distances and ask for confirmation when confidence is low.

In [ ]:
def correct_word(word, word_frequencies=WORD_FREQUENCIES, max_distance=2):
    """Return best dictionary suggestion or the original input. TODO: Exercise 5."""
    # TODO: follow the three ranking/fallback rules in the exercise.
    return word


def check_spelling_correction(correct_fn, *, verbose=True):
    cases = [
        ('کتاب', 'کتاب'),             # already known; don't alter it
        ('دانشجوو', 'دانشجو'),        # one extra character
        ('کتبا', 'کتاب'),             # adjacent transposition costs two edits
        ('يك', 'یک'),                 # normalized form is in the lexicon
        ('ژژژژژ', 'ژژژژژ'),            # no close dictionary candidate; preserve input
    ]
    passed = 0
    for index, (word, expected) in enumerate(cases, 1):
        try:
            got = correct_fn(word)
            ok = got == expected
        except Exception as exc:
            got, ok = repr(exc), False
        passed += int(ok)
        if verbose:
            print(f'{"PASS" if ok else "FAIL"} correction case {index}: {word!r} -> expected {expected!r}, got {got!r}')
    if verbose:
        print(f'{passed}/{len(cases)} correction cases passed')
    return passed, len(cases)

check_spelling_correction(correct_word)

### ✍️ Exercise 5b — Candidate ranking and error analysis

1. Print the top three candidates for `کتبا` and `دانشجوو`, including edit distance and frequency.
2. Construct one typo for which the top suggestion is wrong or ambiguous. Explain why edit distance alone cannot infer intended meaning.
3. Compare `max_distance=1` with `max_distance=2`. Report how coverage and the risk of false corrections change.
4. Explain why a real application should preserve the original text and show a suggestion rather than automatically correcting every token.
5. The toy lexicon is small and hand-curated. How would you audit dictionary quality before deploying a corrector?

---
# 6. Exit ticket

Answer in your own words:

1. Why can stopword removal improve one bag-of-words task but damage sentiment analysis or translation?
2. What is the difference between a stem and a lemma?
3. Why can suffix stripping over-stem Persian words?
4. What three edit operations count toward Levenshtein distance?
5. Why does the corrector rank candidates by frequency only after edit distance?
6. What should happen when no candidate is close enough, and why?
7. Name one limitation of this lab's tokenizer, stopword list, stemmer, lexicon, and spelling corrector.

# Final submission checklist

- [ ] Exercise 0 metadata completed.
- [ ] `remove_stopwords` passes all public checks; negation is preserved.
- [ ] `simple_persian_stem` passes all public checks and its limitations are discussed.
- [ ] `lemmatize_word` uses the supplied lexicon and leaves unknown forms visible.
- [ ] `levenshtein_distance` passes empty-string, normalization, insertion/deletion/substitution, and transposition checks.
- [ ] `correct_word` follows the documented ranking/fallback policy without mutating the dictionary.
- [ ] Written analysis covers why stopword removal is task-dependent and identifies at least one false correction.
- [ ] Restart Kernel and Run All Cells; no unhandled exception remains.
- [ ] Save a private submission as `STUDENT_ID_session04.ipynb`; do not publish a solved notebook publicly.

**End of Session 04.**

## Attribution

- The mini stopword list and mini lexicon in this session are classroom examples created for this lab. They should not be treated as general-purpose lexical resources.